In [ ]:
from astropy.io import fits
import numpy as np
import matplotlib.pyplot as plt
import astroalign as aa

# initialize
R = None
G = None
B = None
L = None
X = None
ref_image = None

for f in light:
    hdr = fits.getheader(f)
    filt = hdr.get('FILTER', 'NOFILTER')

    data = fits.getdata(f)
    data_corr = data - master_bias
    data_reduced = data_corr / master_flats[filt]

    # --- alignment ---
    if ref_image is None:
        ref_image = data_reduced
        aligned = data_reduced
    else:
        try:
            aligned, _ = aa.register(data_reduced, ref_image)
        except Exception as e:
            print(f"Alignment failed for {filt}: {e}")
            continue

    # --- normalization ---
    vmin, vmax = np.percentile(aligned, [1, 99])
    if not np.isfinite(vmin) or not np.isfinite(vmax) or (vmax - vmin) == 0:
        print(f"Skipping {filt} (bad normalization)")
        continue

    norm = (aligned - vmin) / (vmax - vmin)
    norm = np.clip(norm, 0, 1)
    norm = np.nan_to_num(norm, nan=0.0, posinf=0.0, neginf=0.0)

    # --- flexible filter matching ---
    filt_lower = filt.lower()

    if filt_lower.startswith('r'):
        R = norm if R is None else R + norm

    elif filt_lower.startswith('g'):
        G = norm if G is None else G + norm

    elif filt_lower.startswith('b'):
        B = norm if B is None else B + norm

    elif 'ha' in filt_lower or 'h_alpha' in filt_lower:
        if R is None:
            R = norm
        else:
            R = R + 0.5 * norm

    elif 'lum' in filt_lower:
        L = norm

    else:
        print(f"Unknown filter: {filt}")

# --- fallback ---
shape = norm.shape
R = np.zeros(shape) if R is None else R
G = np.zeros(shape) if G is None else G
B = np.zeros(shape) if B is None else B

R *= 2.0
B *= 2.0
G *= 2.0

# --- combine ---
rgb = np.dstack((R, G, B))

# --- luminance ---
if L is not None:
    rgb = rgb * L[:, :, np.newaxis]

# --- clean + normalize ---
rgb = np.nan_to_num(rgb, nan=0.0, posinf=0.0, neginf=0.0)
rgb = rgb - np.min(rgb)

max_val = np.max(rgb)
if max_val > 0:
    rgb = rgb / max_val

# --- plot ---
plt.figure(figsize=(8,6))
plt.imshow(rgb, origin='lower')
plt.axis('off')
plt.title("Aligned Combined RGB image")
plt.show()